Mean shrinkage: James-Stein mean estimator

Reading Material: Before addressing this exercise, I should study Shrinkage "26.a.6"

Code Reference: Become familiar with the script " s_shrinkage_location" (Sample mean shrinkage JS)
which illustrates the James-Stein mean estimator.

Consider the implementation in this example, where Et are the daily log-returns of the stocks in the S&P 500.

Assume the log-returns are truly, normally distributed Et ~ N(Mu, sigma2) i.i.d,

Where:
Mu = alpha1 100x1,  sigma2 = I 100;

for some unknown parameter alpha >= 0.

What this experiment is.
Imagine 100 stocks, each with the same true mean return alpha and variance 1 (independent of each other)
I get only 20 days of data, which is deliberately tight, 100 things to estimate from 20 observations.

Repeat the xperiemnt for 100 different values of alpha from 0 to 1.
Each time estimate the means in 2 ways,
- sample mean mu^, which just averages each stock's 20 returns
- James-Stein mean mu-, takes the sample mean and pulls it to 0.

The theory - why pulling it towards 0 can help.
The sample mean is unbiased, on average its right.  But with only 20 observations its very noisy.
Each stock's estimate has a standard error of 1/square root of 20 ~=0.22.
Squared and added across 100 stocks, thats an expected total squared error of 100/20 = 5

James-Stein makes a trade.
It accepts a small deliberate bias ( a pull towards 0) in exchange for a large cut in noise

In [2]:
from numpy import average, cov, diag, eye, linspace, max, ones, real, trace, zeros
from numpy.linalg import eig, norm
from scipy.stats import multivariate_normal

Task 1.

Define an equally spaced grid {alpha(i)} 100, i=1 of values between [0,1]
and for each i = 1,....,100 perform the following steps:

- Generate t_bar = 20 scenarios {Et}t_bar, t=1 of the daily log-returns Et ~ N(alpha(i) 1 100x1, sigma2).

- Compute the historical mean Mu^(i) and the historical covariance sigma2^(i)
stemming from the time series {Et} t_bar, t=1'

- Compute the  James-Stein mean mu-bar9i) shrinking towards the target vector Mu_target = 0 100x1
and with sub-optimal confidence gamma(i).

Hint: To estimate the sub-optimal confidence gamma(i) replace the unknown population parameters (sigma2, gamma1)
with their historical counterparts(sigma2^(i), gamma^1(i)).

In [5]:
############################ inputs ############################
# number of points on the grid of values for alpha
i_bar = 100
# random vector dimension
n_bar = 100
# number of scenarios
t_bar = 20
################################################################

# 100 equally spaced grid (alpha) values between 0 and 1 (0.000, 0.0101, 0.0202,...)
alpha = linspace(0, 1, i_bar)

# target mean vector (1D - n_bar long)
mu_target = zeros(n_bar)

# population covariance
# using eye() to create a 100x100 identity matrix (variance=1 for each stock and no correlation between them)
sigma2 = eye(n_bar)

#
mu_bar = zeros((i_bar, n_bar))
mu_hat = zeros((i_bar, n_bar))
gamma = zeros((i_bar, n_bar))

# loop
for i in range(0, i_bar):
    # population mean
    mu = alpha[i] * ones(n_bar)
    # scenarios for log-returns, will have a shape of (20,100)
    # one row per day and one column per stock
    eps = multivariate_normal.rvs(mean=mu, cov=sigma2, size=t_bar)
    # historical mean, using average here to give 1 mean per stock
    mu_hat[i, :] = average(eps, axis=0)
    # historical covariance
    # using '.T' here as cov() expects each row to be a variable.
    # bias=True divides t_bar rather than t_bar-1
    sigma2_hat = cov(eps.T, bias=True)
    # eigenvalues and eigenvectors of historical covariance
    lam, e = eig(sigma2_hat)
    # maximum eigenvalue, using real() to strip of tiny parts of eigenvalue
    lam_1 = max(real(lam))
    # sub-optimal confidence
    # dist2 is the squared distance (Mu^ - Mu_target)
    dist2 = (mu_hat[i, :] - mu_target) @ (mu_hat[i, :] - mu_target)
    #
    gamma[i, :] = min(1, 2 / t_bar * (trace(sigma2_hat) - 2 * lam_1) / dist2)
    # shrinkage estimator
    mu_bar[i, :] = (1 - gamma[i, :]) * mu_hat[i, :] + gamma[i, :] * mu_target

# small alpha -> should be 1.0 (full shrink)
print(gamma[:5, 0].round(2))
print()
print(gamma[-5:, 0].round(2))

[1. 1. 1. 1. 1.]

[0.08 0.08 0.08 0.07 0.07]


Task 2:

Compute the sample mean-squared errors 1/100 sum 100, i=1 || Mu^(i) - alpha(i)1 100x1 ||2,2 and 1/100
sum100, i=1 || Mu_hat(i) - alpha(i)1 100x1 || 2,2
to evaluate which mean estimator performs better.

So in a nutshell, compute the average squared error of each estimator across all 100 experiments.

Scoring each of the two estimators, measuring how far each landed from the truth.

using formula:

MSE = 1/100 sum i=1^100||mu^(i) - alpha^(i)1||2^2

alpha^(i)1 = is the true mean vector in experiment i:100 compies of alpha^(i).

mu^(i)-alpha^(i)1 is the miss, one umber per stock

||.||2^2 is the squared Euclidian norm, the same norm as MLFP homework, but squared
Square each stock's miss and add them all up, giving 1 number for the whole experiment.

(1/100) sum averages that across the 100 experiments

This is computed 1 time for Mu_hat(mu^) and 1 time for James-Stein Mu_bar(mu-)


In [9]:
# sum of the Historical mean squared error
sum_mse_mu_hat = 0
# sum of the James-Stein mean squared error
sum_mse_mu_bar = 0

# mu_hat[i, :] = the 100 sample means for experiment i
# alpha[i] * ones(n_bar) = rebuilds the true mean vector for that experiment
# norm(...)**2 is the squared Euclidean norm,  norm() gives the length, **2 squares it.
for i in range(0, i_bar):
    sum_mse_mu_hat = sum_mse_mu_hat + norm(mu_hat[i, :] - alpha[i] * ones(n_bar))**2
    sum_mse_mu_bar = sum_mse_mu_bar + norm(mu_bar[i, :] - alpha[i] * ones(n_bar))**2

# Historical mse
mse_mu_hat = sum_mse_mu_hat / i_bar
# James-Stein mse
mse_mu_bar = sum_mse_mu_bar / i_bar

print("The best estimator is the "+str("James-Stein mean" if mse_mu_hat > mse_mu_bar else "Historical mean"))
print()

############################ results ############################
print("Historical mean squared error:")
print('mse_mu_hat = ', mse_mu_hat.round(3))
print()
print("James-Stein mean squared error:")
print('mse_mu_bar = ', mse_mu_bar.round(3))

The best estimator is the James-Stein mean

Historical mean squared error:
mse_mu_hat =  4.923

James-Stein mean squared error:
mse_mu_bar =  3.826


Conclusion:

The sample mean comes out at almost exactly 5, as per the prediction from Task 1.

Each stocks sample mean has variance 1/20 and with 100 stock that = 5.

James-Stein comes out at ~ 3.8 which ir about 25% lower, accepting some bias but cutting noise.

The improvement gain is not spread evenly across all the experiments:
For a small alpha the truth is near the target of 0.
Gamma~=1 pulls everything to 0, which is almost right, so the error falls off.
For a near 1, the truth is far from the target.
Gamma~=0.07 barely shrinks, so JS performs about the same as the sample mean.

Choosing a sensible target is what makes this work in practice.

When James-Stein is used in practice
1. Expected returns for portfolio optimisation.
2. Ranking many strategies or signals.
3. Alpha signals across a large universe.
4. Evaluating managers and analysts.

Choosing the target

In this homework the target was 0, but in practice the choice of the target is where the skill  lies:
- Zero, for "no edge" beliefs about alpha or excess returns.
- The grand mean, i.e. the average of all the assets. This is the most common choice
- Sector means. ARPM's own example (26.a.22) shrinks each S&P stock towards its MSCI sector index.
- Model-implied values, such as CAPM or equilibrium returns. This ties back to Black-Litterman.

A good target is close to the truth, and thats when James-Stein wins big.


When James-Stein doesnt work.
- Fewer than 3 assets. Stein's result needs at least 3 dimensions.
- Long histories. With lots of data, γ → 0 automatically and it simply reduces to the sample mean.
- A badly chosen target. It still won't lose on average in theory, but the gain shrinks towards nothing.

Shrinkage is Bayesian estimation with the confidence level learned from the data